In [ ]:
from pathlib import Path

# Jupyter normally starts in notebooks/; also support a kernel at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/01_graphs.ipynb"

# 2. Core pruning and random intersection graph generation

This notebook uses the algorithms from the initial implementation: a copied
graph and a deletion queue for core pruning, then two explicit stub lists for
uniform community matching. The cell above loads `Node` and `Graph` from
`01_graphs.ipynb`; subsequent notebooks load this notebook in the same way.
All cells here define reusable code, without running a simulation or plotting.

In [ ]:
import copy
from collections import deque
from collections.abc import Callable, Sequence
from itertools import combinations
from numbers import Integral

import numpy as np

## The $k$-core by repeated deletion

The $k$-core is the maximal induced subgraph in which every vertex has degree at
least $k$. Start with vertices whose degree is below $k$. Delete them one at a
time; whenever a neighbour's degree crosses from $k$ to $k-1$, add it to the
queue. Vertices removed later may already have fallen further below $k$.

`construct_k_core(graph, k)` first deep-copies the graph, so experiments can
inspect both the original graph and its core. The returned graph has the same
UUID labels for surviving vertices and independent node/edge state. Membership
metadata still describes the original random intersection graph.

The queue-based deletion visits each vertex and edge only a bounded number of
times, giving $O(|V|+|E|)$ work, including the graph copy. With $k=0$, all vertices
survive, including isolated vertices.

In [ ]:
def construct_k_core(graph: Graph, k: int) -> Graph:
    """Return the k-core as a separate Graph without modifying the input."""
    if not isinstance(k, Integral) or k < 0:
        raise ValueError("k must be a nonnegative integer")

    core = copy.deepcopy(graph)
    q = deque(node for node in core.nodes if node.degree < k)

    while len(q) > 0:
        node = q.popleft()

        # Removing this node lowers each neighbour's degree by one.
        # Queue exactly the neighbours that are about to cross below k.
        for neighbour in node.neighbours:
            if neighbour.degree == k:
                q.append(neighbour)

        core.remove_node(node)

    return core

## A RIG from fixed-size communities

The generic constructor has the original three required arguments:
`construct_random_intersection_graph(n, k, D)`. Here $n$ is the number of
individuals, $k$ is the fixed number of membership slots per community, and
`D(n)` returns $n$ nonnegative integer membership counts.

1. Sample the **entire** vector $(D_1,\ldots,D_n)$ until its total $M$ is
   divisible by $k$. For independent Poisson draws, this is the joint Poisson
   law conditioned on $M\equiv0\pmod{k}$; changing just one count would give a
   different distribution.
2. Create $M/k$ communities, each with $k$ slots. Build the original explicit
   lists of individual stubs and community slots, shuffle the individual stubs,
   and match the lists position by position.
3. Join each pair of distinct individuals in every community. Repeated slots
   for one individual do not create loops, and repeated projected edges are
   merged. Thus community *size* counts slots and need not equal the number of
   distinct vertices in its projected clique.

The matching is uniform over these labelled stubs. This does **not** mean that
all projected simple graphs are equally likely. The graph retains its accepted
membership counts and every original community slot for later mathematical
analysis.

`rng` is an optional NumPy generator controlling the shuffle. The generic
sampler `D` controls its own randomness; use the same generator in `D` when you
want replayable draws as well. `max_attempts` bounds divisibility rejection so
an incompatible deterministic sampler raises an error instead of looping
forever. The original unconditioned draw-and-retry rule is preserved.

In [ ]:
def construct_random_intersection_graph(
    n: int,
    k: int,
    D: Callable[[int], Sequence[int]],
    rng: np.random.Generator | None = None,
    max_attempts: int = 10_000,
) -> Graph:
    """Construct a simple RIG using the original two-stub-list matching.

    D(n) supplies n membership counts; the complete vector is redrawn until
    its total is divisible by k. The optional rng controls the matching.
    Counts and original community slots are retained on the returned Graph.
    """
    if not isinstance(n, Integral) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    if not isinstance(k, Integral) or k < 1:
        raise ValueError("k must be a positive integer")
    if not isinstance(max_attempts, Integral) or max_attempts < 1:
        raise ValueError("max_attempts must be a positive integer")
    if rng is None:
        rng = np.random.default_rng()

    graph = Graph(n)

    # Rejection conditions the whole vector, never only its final entry.
    for _ in range(max_attempts):
        d = list(D(n))
        if len(d) != n or any(
            not isinstance(count, Integral) or count < 0 for count in d
        ):
            raise ValueError("D(n) must return n nonnegative integer counts")
        d = [int(count) for count in d]
        if sum(d) % k == 0:
            break
    else:
        raise ValueError(
            f"membership total not divisible by {k} after {max_attempts} attempts"
        )

    m = sum(d) // k
    communities = [[] for _ in range(m)]

    # Keep the initial implementation: repeat each Node by its count.
    nodes = list(graph.nodes)
    nodes_by_multiplicity = []
    for i in range(n):
        for _ in range(d[i]):
            nodes_by_multiplicity.append(nodes[i])

    # Repeat the same community list object once for each of its k slots.
    communities_by_multiplicity = []
    for community in communities:
        for _ in range(k):
            communities_by_multiplicity.append(community)

    # One shuffled side suffices for a uniform matching of the two stub lists.
    rng.shuffle(nodes_by_multiplicity)
    for node, community in zip(nodes_by_multiplicity, communities_by_multiplicity):
        community.append(node)

    # Project each community to a clique, suppressing loops and duplicate edges.
    for community in communities:
        for node1, node2 in combinations(community, 2):
            if (node1 != node2) and (node1 not in node2.neighbours):
                graph.add_edge(node1, node2)

    graph.memberships = {node.id: count for node, count in zip(nodes, d)}
    graph.communities = tuple(
        tuple(node.id for node in community) for community in communities
    )
    return graph

## Triangle RIG: Poisson memberships, three slots per community

`construct_special_triangle_RIG(n, pois_param)` is the original convenience
constructor with $D_i\sim\mathrm{Poi}(\lambda)$ before conditioning and $k=3$.
`pois_param` is $\lambda$, the **membership mean**, rather than the mean degree.
The projected mean degree is approximately $2\lambda$ in the sparse regime;
repeated vertices or edges can reduce an individual's projected degree below
twice its accepted membership count.

Pass `rng=np.random.default_rng(seed)` to replay Poisson draws and stub matching.
The UUID labels come from `uuid.uuid4()` and deliberately remain fresh, as in
the initial notebook. A seed reproduces the graph structure **up to relabelling**
and its core size, rather than the UUIDs or the iteration order of node sets.

In [ ]:
def construct_special_triangle_RIG(
    n: int, pois_param: float, rng: np.random.Generator | None = None
) -> Graph:
    """Construct the original Poisson-membership triangle RIG.

    One explicit generator controls both the Poisson counts and the matching.
    Vertex UUID labels are independently generated and are not seed-controlled.
    """
    if not np.isfinite(pois_param) or pois_param < 0:
        raise ValueError("pois_param must be finite and nonnegative")
    if rng is None:
        rng = np.random.default_rng()

    def D(size):
        return rng.poisson(pois_param, size)

    return construct_random_intersection_graph(n, 3, D, rng=rng)